# 🛡️ ShieldBot: Commercial RAG Assistant for Ceylon Shield Insurance
Load every knowledge-base document, parse its YAML front matter into metadata, and check the metadata before building anything on top of it.

In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys

import pandas as pd
from shieldbot import config

print("Python        :", sys.version.split()[0])
print("Project root  :", config.ROOT)
print("KB folder OK  :", config.KB_DIR.exists())
print("Eval CSV OK   :", config.EVAL_CSV.exists())
print("API key loaded:", bool(os.getenv("OPENAI_API_KEY")))
print("LLM model     :", config.LLM_MODEL)

Python        : 3.14.4
Project root  : D:\shieldbot
KB folder OK  : True
Eval CSV OK   : True
API key loaded: True
LLM model     : gpt-4o-mini


## Step 1: Smart ingestion (R1)
Code: `src/shieldbot/ingest.py`. Loads the 26 documents and parses their YAML front matter into metadata.

In [2]:
from shieldbot.ingest import load_documents, metadata_table, check_documents

documents = load_documents(config.KB_DIR)
meta_df = metadata_table(documents)
display(meta_df)

,folder,source_file,doc_id,title,product,doc_type,effective_date,status,audience,chars,version,path,expiry_date,superseded_by,supersedes,confidentiality
0,00_company,branch_directory.md,CS-COM-002,Branch Directory,general,branch_info,2026-06-01,active,public,1397,2026.2,00_company/branch_directory.md,NaN,NaN,NaN,NaN
1,00_company,company_overview.md,CS-COM-001,Company Overview,general,company_info,2026-01-01,active,public,1848,2026.1,00_company/company_overview.md,NaN,NaN,NaN,NaN
2,00_company,contact_and_service_hours.md,CS-COM-003,Contact Channels and Service Hours,general,contact_info,2026-01-01,active,public,1029,2026.1,00_company/contact_and_service_hours.md,NaN,NaN,NaN,NaN
3,01_motor,motor_premium_table_2026.md,CS-MOT-003,Motor Premium Rates 2026,motor,premium_table,2026-01-01,active,public,1421,2026.1,01_motor/motor_premium_table_2026.md,NaN,NaN,NaN,NaN
4,01_motor,motorshield_comprehensive_policy_2026.md,CS-MOT-001,MotorShield Comprehensive — Policy Terms,motor,policy_terms,2026-01-01,active,public,3251,2026.1,01_motor/motorshield_comprehensive_policy_2026.md,NaN,NaN,NaN,NaN
5,01_motor,motorshield_third_party_policy_2026.md,CS-MOT-002,MotorShield Third Party — Policy Terms,motor,policy_terms,2026-01-01,active,public,1098,2026.1,01_motor/motorshield_third_party_policy_2026.md,NaN,NaN,NaN,NaN
6,02_health,health_premium_table_2026.md,CS-HLT-003,Health Shield Plus Premium Table 2026,health,premium_table,2026-01-01,active,public,787,2026.1,02_health/health_premium_table_2026.md,NaN,NaN,NaN,NaN
7,02_health,health_shield_plus_policy_2025.md,CS-HLT-001,Health Shield Plus — Policy Terms (2025),health,policy_terms,2025-01-01,superseded,public,1239,2025.1,02_health/health_shield_plus_policy_2025.md,2025-12-31,CS-HLT-002,NaN,NaN
8,02_health,health_shield_plus_policy_2026.md,CS-HLT-002,Health Shield Plus — Policy Terms (2026),health,policy_terms,2026-01-01,active,public,2607,2026.1,02_health/health_shield_plus_policy_2026.md,NaN,NaN,CS-HLT-001,NaN
9,03_life,life_premium_table.md,CS-LIF-002,Life Secure Premium Table,life,premium_table,2026-01-01,active,public,1026,2026.1,03_life/life_premium_table.md,NaN,NaN,NaN,NaN


In [3]:
print("By status:\n", meta_df["status"].value_counts(), "\n")
print("By audience:\n", meta_df["audience"].value_counts(), "\n")
print("Superseded docs:", meta_df.loc[meta_df["status"] == "superseded", "source_file"].tolist())
print("Internal docs  :", meta_df.loc[meta_df["audience"] == "internal", "source_file"].tolist(), "\n")

check_documents(documents)

By status:
 status
active        25
superseded     1
Name: count, dtype: int64 

By audience:
 audience
public      24
internal     2
Name: count, dtype: int64 

Superseded docs: ['health_shield_plus_policy_2025.md']
Internal docs  : ['agent_handbook.md', 'fraud_awareness_guidelines.md'] 

Step 1 checks passed: 26 docs, 1 superseded, 2 internal, no missing fields


## Step 2: Baseline RAG + evaluation harness (R10)
Code: `src/shieldbot/baseline.py`, `src/shieldbot/evaluation.py`.

A deliberately naive RAG (500-character chunks, vector search only, no metadata filters, no reranking) gives the **"before"** scores. Every later step is measured against it.

In [4]:
from shieldbot.evaluation import load_eval_set

eval_df = load_eval_set(config.EVAL_CSV)
print("CSV columns:", list(eval_df.columns))
display(eval_df[["id", "question", "expected_files"]])

CSV columns: ['id', 'question', 'expected_answer', 'source_documents', 'test_type', 'expected_files']


,id,question,expected_files
0,Q01,What is the waiting period for pre-existing co...,[health_shield_plus_policy_2026.md]
1,Q02,Does the Silver health plan cover maternity?,[health_shield_plus_policy_2026.md]
2,Q03,What is the excess on a motor comprehensive cl...,[motorshield_comprehensive_policy_2026.md]
3,Q04,How much is the comprehensive premium for a pr...,[motor_premium_table_2026.md]
4,Q05,Is the Batticaloa branch open on Saturday?,[branch_directory.md]
5,Q06,I was treated at a non-network hospital. How l...,"[health_claims_procedure.md, claims_overview_p..."
6,Q07,What is the waiting period for the critical il...,[life_secure_plans.md]
7,Q08,I am 35 years old. What is the annual premium ...,[health_premium_table_2026.md]
8,Q09,My baggage was delayed. How much can I claim?,[travel_guard_policy_2026.md]
9,Q10,I am already in Singapore. Can I buy Travel Gu...,[travel_guard_policy_2026.md]


In [5]:
from shieldbot.baseline import BaselineRAG

baseline = BaselineRAG(documents)   # first run downloads bge-small (~130 MB)

# Version trap check: does the superseded 2025 policy show up?
test = baseline.answer("What is the waiting period for pre-existing conditions on Health Shield Plus?")
print(test["answer"], "\n")
for c in test["chunks"]:
    m = c["metadata"]
    print(f"{c['score']:.3f}  {m['source_file']:45} status={m['status']}  audience={m['audience']}")

d:\shieldbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\shieldbot\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Asus\.cache\huggingface\hub\models--BAAI--bge-small-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: http

Baseline index: 107 chunks from 26 docs
The waiting period for pre-existing conditions on Health Shield Plus has been reduced from 24 months to 1 month, effective from the 2026 edition. 

0.801  health_shield_plus_policy_2026.md             status=active  audience=public
0.769  health_shield_plus_policy_2025.md             status=superseded  audience=public
0.759  health_claims_procedure.md                    status=active  audience=public
0.750  health_claims_procedure.md                    status=active  audience=public
0.728  health_premium_table_2026.md                  status=active  audience=public


In [6]:
from shieldbot.evaluation import run_evaluation, summarise

baseline_results = run_evaluation(baseline, eval_df, k=5)
baseline_results.to_csv(config.RESULTS_DIR / "baseline_detail.csv", index=False)   # not committed

baseline_summary = summarise(baseline_results, k=5)
pd.DataFrame([baseline_summary]).to_csv(config.RESULTS_DIR / "baseline_summary.csv", index=False)
pd.Series(baseline_summary)

Q01 ... score 1
Q02 ... score 5
Q03 ... score 5
Q04 ... score 5
Q05 ... score 5
Q06 ... score 5
Q07 ... score 4
Q08 ... score 5
Q09 ... score 5
Q10 ... score 5
Q11 ... score 5
Q12 ... score 5
Q13 ... score 5
Q14 ... score 4
Q15 ... score 5
Q16 ... score 1
Q17 ... score 5
Q18 ... score 5
Q19 ... score 5
Q20 ... score 1


system                        baseline
hit_rate@5                       100.0
mrr                              0.889
avg_correctness                    4.3
guardrail_pass_rate               33.3
questions_leaking_internal           5
questions_using_superseded           4
n_questions                         20
n_retrieval_questions               18
dtype: object

In [7]:
cols = ["id", "hit", "rank", "score", "leaked_internal", "used_superseded", "judge_reason"]
display(baseline_results[cols])

,id,hit,rank,score,leaked_internal,used_superseded,judge_reason
0,Q01,True,1.0,1,False,True,The bot answer provides incorrect information ...
1,Q02,True,2.0,5,False,True,All key facts are correct and nothing importan...
2,Q03,True,1.0,5,True,False,All key facts are correct and nothing importan...
3,Q04,True,1.0,5,False,False,All key facts are correct and nothing importan...
4,Q05,True,1.0,5,True,False,All key facts are correct and nothing importan...
5,Q06,True,1.0,5,False,True,All key facts are correct and nothing importan...
6,Q07,True,2.0,4,False,True,The bot answer is correct but omits the 30-day...
7,Q08,True,1.0,5,False,False,All key facts are correct and nothing importan...
8,Q09,True,1.0,5,False,False,All key facts are correct and nothing importan...
9,Q10,True,2.0,5,False,False,All key facts are correct and nothing importan...
